# Ver as leis

Só carrega as leis de `coleta/` para dar uma olhada, sem modelo e sem GPU: roda em qualquer ambiente do Colab.

Mostra quantas leis há por ano, a amostra fixa de 300 leis (a mesma que os notebooks do Gemma, do Llama e do Qwen processam) e, para uma lei escolhida, o texto como o modelo recebe, a página renderizada e o JSON do regex.

In [ ]:
%cd /content
!test -d extrator_fragmentos_leis || git clone --depth 1 https://github.com/filipemcbarros/extrator_fragmentos_leis.git
!pip install -q lxml

In [ ]:
import json, pathlib, random, re
import pandas as pd
from lxml import etree, html as lhtml
from IPython.display import HTML, display

COLETA = pathlib.Path("/content/extrator_fragmentos_leis/coleta")
LIMITE = 300   # o mesmo LIMITE dos notebooks dos modelos

leis = sorted(COLETA.glob("*/html/*.html"))
# A mesma amostra dos notebooks dos modelos: sorteio com semente 0 sobre o acervo inteiro.
amostra = random.Random(0).sample(leis, min(LIMITE, len(leis)))

tabela = pd.DataFrame({
    "slug": [a.stem for a in leis],
    "ano": [a.parent.parent.name for a in leis],
    "kb": [round(a.stat().st_size / 1024, 1) for a in leis],
})
tabela["na_amostra"] = tabela["slug"].isin({a.stem for a in amostra})
print(f"{len(leis)} leis no acervo, {len(amostra)} na amostra.")
tabela.groupby("ano").agg(leis=("slug", "size"), na_amostra=("na_amostra", "sum"))

## A amostra de 300 leis

In [ ]:
pd.set_option("display.max_rows", 400)
tabela[tabela["na_amostra"]].reset_index(drop=True)

## Todas as leis num DataFrame

Uma lei por linha: título, ementa, preâmbulo, fecho e contagens de dispositivos (do JSON do regex), o texto puro da lei e se ela está na amostra de 300. Leva cerca de um minuto para as 16 mil leis.

In [ ]:
def texto_da_lei(arquivo):
    """O texto puro do bloco da lei, sem as tags HTML."""
    pagina = lhtml.fromstring(arquivo.read_text(encoding="utf-8"))
    bloco = pagina.xpath('//div[contains(@class,"textoNorma")]')
    return " ".join(bloco[0].text_content().split()) if bloco else ""

na_amostra = {a.stem for a in amostra}
linhas = []
for arquivo in leis:
    ano = arquivo.parent.parent.name
    regex = json.loads((COLETA / ano / "json" / f"{arquivo.stem}.json").read_text(encoding="utf-8"))
    linhas.append({
        "slug": arquivo.stem,
        "ano": int(ano),
        "titulo": regex.get("titulo"),
        "ementa": regex.get("ementa"),
        "preambulo": regex.get("preambulo"),
        "fecho": regex.get("fecho"),
        "origem": regex.get("origem"),
        **{f"n_{tipo}": n for tipo, n in regex.get("contagem", {}).items()},
        "na_amostra": arquivo.stem in na_amostra,
        "texto": texto_da_lei(arquivo),
    })

df = pd.DataFrame(linhas)
contagens = sorted(c for c in df.columns if c.startswith("n_"))
df[contagens] = df[contagens].fillna(0).astype(int)
df = df[[c for c in df.columns if c not in contagens + ["texto"]] + contagens + ["texto"]]
print(df.shape)
df.head()

## Uma lei

Troque `SLUG` pelo nome de uma lei da tabela acima (ou deixe `None` para sortear uma da amostra). Saem o texto recortado que vai para o modelo, a página original e o JSON que o regex extraiu.

In [ ]:
SLUG = None

def recortar(arquivo):
    """Título e bloco da lei, como em fragmentar.py: o que o modelo recebe."""
    pagina = lhtml.fromstring(arquivo.read_text(encoding="utf-8"))
    titulos = (" ".join(h.text_content().split()) for h in pagina.xpath("//h1"))
    titulo = next((t for t in titulos if any(c.isdigit() for c in t)), "")
    bloco = pagina.xpath('//div[contains(@class,"textoNorma")]')[0]
    etree.strip_elements(bloco, etree.Comment, "script", "style", with_tail=False)
    for e in bloco.iter():
        classe = e.get("class")
        e.attrib.clear()
        if classe:
            e.set("class", classe)
    texto = lhtml.tostring(bloco, encoding="unicode").replace("\xa0", " ")
    texto = re.sub(r"\s*\n\s*", "\n", re.sub(r"[ \t\r]+", " ", texto)).strip()
    return f"<h1>{titulo}</h1>\n{texto}"

arquivo = next(a for a in leis if a.stem == SLUG) if SLUG else random.choice(amostra)
lei = recortar(arquivo)
print(arquivo.stem, "-", arquivo.parent.parent.name, f"- {len(lei)} caracteres")
print(lei[:5000])

In [ ]:
# A lei renderizada, como na página.
display(HTML(f'<div style="max-height:600px;overflow:auto;border:1px solid #ccc;padding:8px">{lei}</div>'))

In [ ]:
# O que o regex extraiu desta lei.
do_regex = COLETA / arquivo.parent.parent.name / "json" / f"{arquivo.stem}.json"
if do_regex.exists():
    dados = json.loads(do_regex.read_text(encoding="utf-8"))
    print("contagem:", dados.get("contagem"))
    print(json.dumps(dados, ensure_ascii=False, indent=2)[:4000])
else:
    print("Sem JSON do regex para esta lei.")

## Distribuição de tokens

Conta os tokens de cada lei com o tokenizador do Gemma 3, o mesmo que o `gemma_a100.ipynb` usa para os cortes de `TRECHO = 3000` e `MAX_TOKENS_LEI = 9000`. Conta duas versões: `tokens_html`, o HTML recortado que o modelo recebe (a mesma função `recortar` de cima), e `tokens_texto`, o texto puro. Precisa do `HF_TOKEN` nos Secrets e da licença do Gemma aceita, e das células do DataFrame e de "Uma lei" já rodadas.

In [ ]:
!pip install -q transformers matplotlib
import matplotlib.pyplot as plt
import numpy as np
from google.colab import userdata
from transformers import AutoTokenizer

tokenizador = AutoTokenizer.from_pretrained("google/gemma-3-12b-it", token=userdata.get("HF_TOKEN"))

def contar(textos):
    return [len(ids) for ids in tokenizador(textos, add_special_tokens=False)["input_ids"]]

caminho = {a.stem: a for a in leis}
df["tokens_html"] = contar([recortar(caminho[s]) for s in df["slug"]])
df["tokens_texto"] = contar(df["texto"].tolist())

# O que acontece com cada lei nos notebooks dos modelos.
df["faixa"] = pd.cut(df["tokens_html"], [0, 3000, 9000, float("inf")],
                     labels=["inteira (até 3000)", "dividida em trechos", "fica de fora (> 9000)"])

display(df[["tokens_html", "tokens_texto"]].describe(percentiles=[.5, .75, .9, .95, .99]).round(0))
display(pd.crosstab(df["faixa"], df["na_amostra"], margins=True))
print(f"Total: {df.tokens_html.sum():,} tokens de HTML, {df.tokens_texto.sum():,} de texto puro.")

fig, eixos = plt.subplots(1, 2, figsize=(13, 4))
for eixo, coluna in zip(eixos, ["tokens_html", "tokens_texto"]):
    eixo.hist(df[coluna], bins=np.logspace(1, np.log10(df[coluna].max() + 1), 60))
    eixo.set_xscale("log")
    for corte in (3000, 9000):
        eixo.axvline(corte, color="red", linestyle="--")
    eixo.set_title(coluna)
    eixo.set_xlabel("tokens (escala log)")
    eixo.set_ylabel("leis")
plt.tight_layout()
plt.show()